# 06 — Évaluation Out-of-Domain : LIAR → WELFake

## Objectif

Tester la **capacité de généralisation** des modèles entraînés sur LIAR sur le dataset WELFake (Verma et al., 2021).

## Dataset externe : WELFake

| | LIAR (entraînement) | WELFake (évaluation) |
|---|---|---|
| Source | PolitiFact | Kaggle + McIntire + Reuters + BuzzFeed |
| Type de texte | Déclaration (~18 mots) | Article complet (~540 mots) |
| Labels | 3 classes (fake/nuanced/real) | Binaire (0=fake, 1=real) |
| Taille | 10 240 (train) | 72 134 |
| Métadonnées speaker | **Oui** (credibility, lie_rate…) | **Non** |

## Hypothèses à tester

1. **Les modèles avec métadonnées vont chuter plus que le modèle texte seul** — credibility_score et lie_rate sont à 0 pour tous les articles WELFake.
2. **La chute sera plus forte sur la classe nuanced** — absente de WELFake, les modèles n'auront aucun exemple de référence.
3. **DistilBERT devrait mieux généraliser que RF** — il s'appuie uniquement sur le texte.

## 0. Imports

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from preprocessing import load_liar, preprocess_liar
from features import TfidfFeatures, CombinedFeatures, META_COLS
from train import train_logistic_regression, train_random_forest, train_xgboost
from evaluate import evaluate_model
from welfake_loader import (
    load_welfake, preprocess_welfake,
    add_dummy_metadata, print_domain_shift_summary,
)

## 1. Entraînement des modèles sur LIAR

In [2]:
train_raw, valid_raw, test_raw = load_liar('data/raw')
train = preprocess_liar(train_raw)
test  = preprocess_liar(test_raw)

y_train = train['label_encoded'].values
y_test  = test['label_encoded'].values

# Features avec métadonnées
tfidf    = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf.fit_transform(train['statement_clean'])
combined = CombinedFeatures(text_features=tfidf, meta_cols=META_COLS)
X_train  = combined.fit_transform(train)
X_test   = combined.transform(test)

# Features texte seul (pour comparaison section 7)
X_train_text = tfidf.transform(train['statement_clean'])
X_test_text  = tfidf.transform(test['statement_clean'])

print(f'X_train (avec meta) : {X_train.shape}')
print(f'X_train (texte seul): {X_train_text.shape}')

LIAR chargé — train: 10240, valid: 1284, test: 1267
TF-IDF fit — vocabulaire : 9049 features, 10240 documents
X_train (avec meta) : (10240, 9053)
X_train (texte seul): (10240, 9049)


In [3]:
lr      = train_logistic_regression(X_train, y_train)
rf      = train_random_forest(X_train, y_train, n_estimators=300)
xgb     = train_xgboost(X_train, y_train)
rf_text = train_random_forest(X_train_text, y_train, n_estimators=300)
print('Tous les modèles entraînés.')

Entraînement Logistic Regression...


  LR — terminé
Entraînement Random Forest...
  RF — terminé
Entraînement XGBoost...
  XGBoost — terminé
Entraînement Random Forest...
  RF — terminé
Tous les modèles entraînés.


## 2. Scores in-domain (référence LIAR)

In [4]:
results_indomain = []
results_indomain.append(evaluate_model(y_test, lr.predict(X_test),  'LR'))
results_indomain.append(evaluate_model(y_test, rf.predict(X_test),  'RF'))
X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else X_test
results_indomain.append(evaluate_model(y_test, xgb.predict(X_test_dense), 'XGBoost'))
res_rf_text_in = evaluate_model(y_test, rf_text.predict(X_test_text), 'RF texte seul')


  LR
  Accuracy    : 0.5375
  F1 macro    : 0.5442   ← métrique principale
  F1 weighted : 0.5338

  F1 par classe :
    fake       : 0.6465
    nuanced    : 0.4705
    real       : 0.5156

              precision    recall  f1-score   support

        fake       0.61      0.69      0.65       341
     nuanced       0.51      0.43      0.47       477
        real       0.50      0.53      0.52       449

    accuracy                           0.54      1267
   macro avg       0.54      0.55      0.54      1267
weighted avg       0.53      0.54      0.53      1267


  RF
  Accuracy    : 0.5651
  F1 macro    : 0.5697   ← métrique principale
  F1 weighted : 0.5616

  F1 par classe :
    fake       : 0.6431
    nuanced    : 0.4874
    real       : 0.5786

              precision    recall  f1-score   support

        fake       0.62      0.67      0.64       341
     nuanced       0.54      0.45      0.49       477
        real       0.55      0.61      0.58       449

    accuracy       

## 3. Chargement et preprocessing WELFake

In [5]:
# sample_n=5000 => 5000 fake + 5000 real = 10 000 articles
# Mettre None pour tout le dataset (72k, plus lent)
welfake = load_welfake(
    path='data/external/WELFake_Dataset.csv',
    sample_n=5000,
    random_state=42,
)

WELFake chargé — 10000 articles
Distribution des labels :
  fake       :   5000  (50.0%)
  real       :   5000  (50.0%)


In [6]:
welfake   = preprocess_welfake(welfake, max_chars=500)
welfake   = add_dummy_metadata(welfake)
y_welfake = welfake['label_encoded'].values
print(f'Classes présentes : {np.unique(y_welfake)} (0=fake, 2=real)')

  14 textes vides après nettoyage
Preprocessing terminé — longueur moy. après nettoyage : 51 mots
Classes présentes : [0 2] (0=fake, 2=real)


In [7]:
print_domain_shift_summary(test, welfake)

  ANALYSE DU DOMAIN SHIFT : LIAR vs WELFake

Caractéristique                    LIAR    WELFake
--------------------------------------------------
  Taille (n exemples)              1267      10000
  Longueur moy. (mots)             11.7       51.0
  Longueur max (mots)               327         72
  Type de texte              déclaration    article
  Métadonnées speaker               oui        non
  Nb de classes                       3          2

  Distribution des classes :
    fake         : LIAR  26.9%   WELFake  50.0%
    nuanced      : LIAR  37.6%   WELFake   0.0%
    real         : LIAR  35.4%   WELFake  50.0%



## 4. Construction des features WELFake

On applique le TF-IDF **appris sur LIAR** (`.transform()` seulement).  
C'est le cœur du test : est-ce que le vocabulaire de déclarations politiques courtes capture quelque chose dans des articles complets ?

In [8]:
X_welfake      = combined.transform(welfake)
X_welfake_text = tfidf.transform(welfake['statement_clean'])

nonzero = (X_welfake_text.sum(axis=1) > 0).sum()
print(f'Couverture vocab LIAR sur WELFake : {nonzero}/{len(welfake)} ({nonzero/len(welfake)*100:.1f}%)')

density_liar    = X_test_text.nnz / (X_test_text.shape[0] * X_test_text.shape[1])
density_welfake = X_welfake_text.nnz / (X_welfake_text.shape[0] * X_welfake_text.shape[1])
print(f'Densité TF-IDF — LIAR: {density_liar*100:.4f}%  |  WELFake: {density_welfake*100:.4f}%')

Couverture vocab LIAR sur WELFake : 9973/10000 (99.7%)
Densité TF-IDF — LIAR: 0.1366%  |  WELFake: 0.3667%


## 5. Évaluation out-of-domain

In [9]:
results_outdomain = []
results_outdomain.append(evaluate_model(y_welfake, lr.predict(X_welfake),  'LR'))
results_outdomain.append(evaluate_model(y_welfake, rf.predict(X_welfake),  'RF'))
X_welfake_dense = X_welfake.toarray() if hasattr(X_welfake, 'toarray') else X_welfake
results_outdomain.append(evaluate_model(y_welfake, xgb.predict(X_welfake_dense), 'XGBoost'))
res_rf_text_out = evaluate_model(y_welfake, rf_text.predict(X_welfake_text), 'RF texte seul')


  LR
  Accuracy    : 0.0676
  F1 macro    : 0.0682   ← métrique principale
  F1 weighted : 0.1022

  F1 par classe :
    fake       : 0.0000
    nuanced    : 0.0000
    real       : 0.2045

              precision    recall  f1-score   support

        fake       0.00      0.00      0.00      5000
     nuanced       0.00      0.00      0.00         0
        real       0.42      0.14      0.20      5000

    accuracy                           0.07     10000
   macro avg       0.14      0.05      0.07     10000
weighted avg       0.21      0.07      0.10     10000


  RF
  Accuracy    : 0.5000
  F1 macro    : 0.3333   ← métrique principale
  F1 weighted : 0.3333

  F1 par classe :
    fake       : 0.0000
    nuanced    : 0.0000
    real       : 0.6667

              precision    recall  f1-score   support

        fake       0.00      0.00      0.00      5000
     nuanced       0.00      0.00      0.00         0
        real       0.50      1.00      0.67      5000

    accuracy       

## 6. Analyse de la chute de performance

In [10]:
df_in  = pd.DataFrame(results_indomain)
df_out = pd.DataFrame(results_outdomain)

print(f'{"Modèle":<16} | {"In-domain":>10} | {"Out-domain":>10} | {"Chute":>8}')
print('-' * 53)
for m in ['LR', 'RF', 'XGBoost']:
    f1_in  = df_in[df_in['model'] == m]['f1_macro'].values[0]
    f1_out = df_out[df_out['model'] == m]['f1_macro'].values[0]
    print(f'  {m:<14} | {f1_in:>10.4f} | {f1_out:>10.4f} | {(f1_out-f1_in)*100:>+7.2f} pp')

print()
print('Impact des métadonnées sur RF :')
rf_in_f1  = df_in[df_in['model']=='RF']['f1_macro'].values[0]
rf_out_f1 = df_out[df_out['model']=='RF']['f1_macro'].values[0]
print(f'  RF avec méta  — in:{rf_in_f1:.4f}  out:{rf_out_f1:.4f}  chute:{(rf_out_f1-rf_in_f1)*100:+.2f} pp')
print(f"  RF texte seul — in:{res_rf_text_in['f1_macro']:.4f}  out:{res_rf_text_out['f1_macro']:.4f}  chute:{(res_rf_text_out['f1_macro']-res_rf_text_in['f1_macro'])*100:+.2f} pp")

Modèle           |  In-domain | Out-domain |    Chute
-----------------------------------------------------
  LR             |     0.5442 |     0.0682 |  -47.60 pp
  RF             |     0.5697 |     0.3333 |  -23.64 pp
  XGBoost        |     0.5577 |     0.3339 |  -22.38 pp

Impact des métadonnées sur RF :
  RF avec méta  — in:0.5697  out:0.3333  chute:-23.64 pp
  RF texte seul — in:0.4300  out:0.1819  chute:-24.81 pp


In [11]:
models  = ['LR', 'RF', 'XGBoost']
metrics = ['f1_macro', 'f1_fake', 'f1_real']
labels  = ['F1 macro', 'F1 fake', 'F1 real']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, m in zip(axes, models):
    row_in  = df_in[df_in['model'] == m].iloc[0]
    row_out = df_out[df_out['model'] == m].iloc[0]
    x, w = np.arange(len(metrics)), 0.35
    b_in  = ax.bar(x - w/2, [row_in[k]  for k in metrics], w, label='In-domain (LIAR)',       color='#378ADD', alpha=0.85)
    b_out = ax.bar(x + w/2, [row_out[k] for k in metrics], w, label='Out-of-domain (WELFake)', color='#D85A30', alpha=0.85)
    for bar in list(b_in) + list(b_out):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(m, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('In-domain (LIAR) vs Out-of-domain (WELFake)', fontsize=13)
plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/domain_shift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée -> outputs/figures/domain_shift_comparison.png')

Figure sauvegardée -> outputs/figures/domain_shift_comparison.png


## 7. Impact des métadonnées sur la généralisation

In [12]:
fig, ax = plt.subplots(figsize=(9, 4))
configs = [
    ('RF avec méta',   rf_in_f1,                    rf_out_f1),
    ('RF texte seul',  res_rf_text_in['f1_macro'],  res_rf_text_out['f1_macro']),
]
x, w = np.arange(len(configs)), 0.35
b_in  = ax.bar(x - w/2, [c[1] for c in configs], w, label='In-domain (LIAR)',       color='#378ADD', alpha=0.85)
b_out = ax.bar(x + w/2, [c[2] for c in configs], w, label='Out-of-domain (WELFake)', color='#D85A30', alpha=0.85)
for bar in list(b_in) + list(b_out):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}', ha='center', va='bottom', fontsize=9)
for i, (name, f_in, f_out) in enumerate(configs):
    drop = f_out - f_in
    ax.annotate(f'Δ = {drop*100:+.1f} pp', xy=(i + w/2, f_out + 0.05),
                ha='center', fontsize=10, color='#c0392b' if drop < 0 else '#27ae60', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c[0] for c in configs], fontsize=11)
ax.set_ylim(0, 0.95)
ax.set_ylabel('F1 macro')
ax.set_title('Impact des métadonnées sur la généralisation (RF)', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/metadata_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée -> outputs/figures/metadata_impact.png')

Figure sauvegardée -> outputs/figures/metadata_impact.png


## 8. Sauvegarde

In [13]:
rows = []
for r_in, r_out in zip(results_indomain, results_outdomain):
    m = r_in['model']
    rows.append({'model': m, 'domain': 'in-domain (LIAR)',       **{k: r_in[k]  for k in ['accuracy','f1_macro','f1_fake','f1_nuanced','f1_real']}})
    rows.append({'model': m, 'domain': 'out-of-domain (WELFake)', **{k: r_out[k] for k in ['accuracy','f1_macro','f1_fake','f1_nuanced','f1_real']}})
for r_in, r_out in [(res_rf_text_in, res_rf_text_out)]:
    m = r_in['model']
    rows.append({'model': m, 'domain': 'in-domain (LIAR)',       **{k: r_in[k]  for k in ['accuracy','f1_macro','f1_fake','f1_nuanced','f1_real']}})
    rows.append({'model': m, 'domain': 'out-of-domain (WELFake)', **{k: r_out[k] for k in ['accuracy','f1_macro','f1_fake','f1_nuanced','f1_real']}})

df_final = pd.DataFrame(rows)
df_final.to_csv('outputs/results_domain_shift.csv', index=False)
print('Résultats sauvegardés -> outputs/results_domain_shift.csv')
df_final

Résultats sauvegardés -> outputs/results_domain_shift.csv


,model,domain,accuracy,f1_macro,f1_fake,f1_nuanced,f1_real
0,LR,in-domain (LIAR),0.5375,0.5442,0.6465,0.4705,0.5156
1,LR,out-of-domain (WELFake),0.0676,0.0682,0.0000,0.0000,0.2045
2,RF,in-domain (LIAR),0.5651,0.5697,0.6431,0.4874,0.5786
3,RF,out-of-domain (WELFake),0.5000,0.3333,0.0000,0.0000,0.6667
4,XGBoost,in-domain (LIAR),0.5556,0.5577,0.6239,0.4824,0.5668
5,XGBoost,out-of-domain (WELFake),0.5001,0.3339,0.0012,0.0000,0.6666
6,RF texte seul,in-domain (LIAR),0.4428,0.4300,0.3584,0.4082,0.5233
7,RF texte seul,out-of-domain (WELFake),0.1943,0.1819,0.2703,0.0000,0.2752


## 9. Visualisations interactives

Quatre graphiques interactifs pour explorer les résultats :

1. **Grouped bar — F1 macro** : comparaison in vs out par modèle  
2. **Chute en points de pourcentage** : classement des modèles par résistance au domain shift  
3. **F1 par classe** : radar / barres groupées fake / nuanced / real  
4. **Heatmap** : toutes les métriques × tous les modèles × les deux domaines

In [14]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

# Reconstruction du DataFrame depuis les résultats déjà calculés
records_in = []
records_out = []

for r_in, r_out in zip(results_indomain, results_outdomain):
    records_in.append(r_in)
    records_out.append(r_out)

records_in.append(res_rf_text_in)
records_out.append(res_rf_text_out)

df_in_plot  = pd.DataFrame(records_in)
df_out_plot = pd.DataFrame(records_out)

MODELS = df_in_plot["model"].tolist()
print("Modèles disponibles :", MODELS)
print("Données prêtes pour Plotly.")

Modèles disponibles : ['LR', 'RF', 'XGBoost', 'RF texte seul']
Données prêtes pour Plotly.


In [15]:
# --- Graphique 1 : F1 macro In-domain vs Out-of-domain ---

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    name="In-domain (LIAR)",
    x=MODELS,
    y=df_in_plot["f1_macro"].tolist(),
    marker_color="#378ADD",
    opacity=0.85,
    text=[f"{v:.3f}" for v in df_in_plot["f1_macro"]],
    textposition="outside",
))

fig1.add_trace(go.Bar(
    name="Out-of-domain (WELFake)",
    x=MODELS,
    y=df_out_plot["f1_macro"].tolist(),
    marker_color="#D85A30",
    opacity=0.85,
    text=[f"{v:.3f}" for v in df_out_plot["f1_macro"]],
    textposition="outside",
))

fig1.update_layout(
    title=dict(text="F1 macro — In-domain (LIAR) vs Out-of-domain (WELFake)", font=dict(size=16)),
    barmode="group",
    yaxis=dict(title="F1 macro", range=[0, 0.85], tickformat=".2f"),
    xaxis_title="Modèle",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
    height=480,
)

fig1.show()
print("Graphique 1 — F1 macro In vs Out affiché.")

Graphique 1 — F1 macro In vs Out affiché.


In [16]:
# --- Graphique 2 : Chute en points de pourcentage ---

drops_pp = [
    (m, (f_out - f_in) * 100)
    for m, f_in, f_out in zip(
        MODELS,
        df_in_plot["f1_macro"].tolist(),
        df_out_plot["f1_macro"].tolist(),
    )
]
drops_pp_sorted = sorted(drops_pp, key=lambda x: x[1], reverse=True)  # moins de chute en premier

model_labels = [d[0] for d in drops_pp_sorted]
drop_values  = [d[1] for d in drops_pp_sorted]
bar_colors   = ["#27ae60" if v >= 0 else "#c0392b" for v in drop_values]

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=drop_values,
    y=model_labels,
    orientation="h",
    marker_color=bar_colors,
    opacity=0.85,
    text=[f"{v:+.1f} pp" for v in drop_values],
    textposition="outside",
))

fig2.add_vline(x=0, line_color="black", line_width=1.5)

fig2.update_layout(
    title=dict(text="Chute de performance In-domain → Out-of-domain (F1 macro, en pp)", font=dict(size=16)),
    xaxis=dict(title="Chute (points de pourcentage)", tickformat="+.0f", range=[-60, 5]),
    yaxis_title="Modèle",
    template="plotly_white",
    height=380,
    margin=dict(l=120),
)

fig2.show()
print("Graphique 2 — Chute en pp affiché.")

Graphique 2 — Chute en pp affiché.


In [17]:
# --- Graphique 3 : F1 par classe (fake / nuanced / real) — subplots par modèle ---

CLASS_METRICS = ["f1_fake", "f1_nuanced", "f1_real"]
CLASS_LABELS  = ["fake", "nuanced", "real"]
CLASS_COLORS  = ["#d73027", "#fdae61", "#1a9850"]

fig3 = make_subplots(
    rows=1, cols=len(MODELS),
    subplot_titles=MODELS,
    shared_yaxes=True,
)

for col_idx, model in enumerate(MODELS, start=1):
    row_in  = df_in_plot[df_in_plot["model"] == model].iloc[0]
    row_out = df_out_plot[df_out_plot["model"] == model].iloc[0]

    vals_in  = [row_in[m]  for m in CLASS_METRICS]
    vals_out = [row_out[m] for m in CLASS_METRICS]

    showlegend = col_idx == 1

    for i, (cls, color) in enumerate(zip(CLASS_LABELS, CLASS_COLORS)):
        # In-domain : barre pleine
        fig3.add_trace(go.Bar(
            name=f"{cls} (in)",
            x=[cls],
            y=[vals_in[i]],
            marker_color=color,
            opacity=0.9,
            showlegend=False,
            legendgroup=f"in_{cls}",
            text=[f"{vals_in[i]:.2f}"],
            textposition="outside",
        ), row=1, col=col_idx)

        # Out-of-domain : même couleur, motif hachuré
        fig3.add_trace(go.Bar(
            name=f"{cls} (out)",
            x=[cls],
            y=[vals_out[i]],
            marker=dict(color=color, opacity=0.45, pattern=dict(shape="/")),
            showlegend=False,
            legendgroup=f"out_{cls}",
            text=[f"{vals_out[i]:.2f}"],
            textposition="outside",
        ), row=1, col=col_idx)

# Ajout de traces fantômes pour la légende
for lbl, color, pattern in [
    ("In-domain (LIAR)", "#555555", None),
    ("Out-of-domain (WELFake)", "#555555", dict(shape="/")),
]:
    fig3.add_trace(go.Bar(
        name=lbl,
        x=[None], y=[None],
        marker=dict(color=color, pattern=pattern) if pattern else dict(color=color),
        showlegend=True,
    ))

fig3.update_layout(
    title=dict(text="F1 par classe — In-domain vs Out-of-domain", font=dict(size=16)),
    barmode="group",
    yaxis=dict(range=[0, 1.0], title="F1", tickformat=".2f"),
    template="plotly_white",
    height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
)

fig3.show()
print("Graphique 3 — F1 par classe affiché.")

Graphique 3 — F1 par classe affiché.


In [18]:
# --- Graphique 4 : Heatmap toutes métriques × modèles × domaines ---

ALL_METRICS   = ["accuracy", "f1_macro", "f1_fake", "f1_nuanced", "f1_real"]
METRIC_LABELS = ["Accuracy", "F1 macro", "F1 fake", "F1 nuanced", "F1 real"]

# Construction d'une matrice : lignes = (modèle + domaine), colonnes = métriques
row_labels = []
z_matrix   = []

for model in MODELS:
    for (df_dom, dom_label) in [(df_in_plot, "in-domain"), (df_out_plot, "out-of-domain")]:
        row = df_dom[df_dom["model"] == model].iloc[0]
        row_labels.append(f"{model} ({dom_label})")
        z_matrix.append([round(row[m], 3) for m in ALL_METRICS])

fig4 = go.Figure(data=go.Heatmap(
    z=z_matrix,
    x=METRIC_LABELS,
    y=row_labels,
    colorscale="RdYlGn",
    zmin=0.0,
    zmax=0.75,
    text=[[f"{v:.3f}" for v in row] for row in z_matrix],
    texttemplate="%{text}",
    textfont=dict(size=11),
    showscale=True,
    colorbar=dict(title="Score"),
))

fig4.update_layout(
    title=dict(text="Heatmap — Toutes métriques × Modèles × Domaines", font=dict(size=16)),
    xaxis_title="Métrique",
    yaxis=dict(autorange="reversed", title="Modèle × Domaine"),
    template="plotly_white",
    height=520,
    margin=dict(l=200),
)

fig4.show()
print("Graphique 4 — Heatmap affichée.")

Graphique 4 — Heatmap affichée.
